# Import, configuration and paths

In [1]:
from pathlib import Path
import json
import pickle
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# Paths
# ============================================================

PROCESSED_DATA_DIR = Path("../Data/Processed")

TRAIN_PATH = PROCESSED_DATA_DIR / "train_air_quality.parquet"
VAL_PATH = PROCESSED_DATA_DIR / "validation_air_quality.parquet"
TEST_PATH = PROCESSED_DATA_DIR / "test_air_quality.parquet"
SPLIT_METADATA_PATH = PROCESSED_DATA_DIR / "split_metadata.json"

PREPROCESSOR_PATH = PROCESSED_DATA_DIR / "preprocessing_metadata.pkl"

# ============================================================
# Forecasting Configuration
# ============================================================

ENCODER_LENGTH = 168
PREDICTION_LENGTH = 15

TARGET_COLUMNS = [
    "pm2_5",
    "pm10",
    "no",
    "no2",
    "nox",
    "nh3",
    "co",
    "so2",
    "o3",
]

WEATHER_COLUMNS = [
    "ambient_temperature",
    "relative_humidity",
    "solar_radiation",
    "rainfall",
]

REQUIRED_COLUMNS = [
    "timestamp",
    "station_id",
    "time_idx",
    "sequence_segment_id",
    *TARGET_COLUMNS,
]

print("Preprocessing configuration initialized.")
print(f"Encoder length    : {ENCODER_LENGTH} hours")
print(f"Prediction length : {PREDICTION_LENGTH} hours")
print(f"Targets           : {len(TARGET_COLUMNS)}")

Preprocessing configuration initialized.
Encoder length    : 168 hours
Prediction length : 15 hours
Targets           : 9


# Load split and validate previous 

In [2]:
# ============================================================
# Load Data Splits
# ============================================================

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH, SPLIT_METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Run 05_data_splitting.ipynb first."
        )

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

with open(SPLIT_METADATA_PATH, "r") as file:
    split_metadata = json.load(file)

# ============================================================
# Basic Validation
# ============================================================

for split_name, df in {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df,
}.items():

    missing_columns = [
        column for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{split_name} split missing columns: {missing_columns}"
        )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="raise",
    )

    df.sort_values(
        ["station_id", "sequence_segment_id", "timestamp"],
        inplace=True,
    )

    df.reset_index(drop=True, inplace=True)

print("Dataset splits loaded and validated.\n")

for split_name, df in {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df,
}.items():

    print(
        f"{split_name:<10} | "
        f"Rows: {len(df):>10,} | "
        f"Stations: {df['station_id'].nunique():>3} | "
        f"{df['timestamp'].min()} -> {df['timestamp'].max()}"
    )

assert train_df["timestamp"].max() < val_df["timestamp"].min()
assert val_df["timestamp"].max() < test_df["timestamp"].min()

print("\nChronological split validation passed.")

Dataset splits loaded and validated.

Train      | Rows:  2,007,399 | Stations:  66 | 2017-01-01 00:00:00 -> 2023-08-25 13:00:00
Validation | Rows:    747,056 | Stations:  65 | 2023-08-25 14:00:00 -> 2025-01-26 18:00:00
Test       | Rows:    674,665 | Stations:  64 | 2025-01-26 19:00:00 -> 2026-06-30 23:00:00

Chronological split validation passed.


# Detwct Model Feature and prepare data types

In [3]:
# ============================================================
# Define Feature Groups
# ============================================================

EXCLUDED_COLUMNS = {
    "timestamp",
    "station_id",
    "state",
    "station",
    "sequence_segment_id",
}

NUMERIC_FEATURE_COLUMNS = [
    column
    for column in train_df.select_dtypes(include=[np.number]).columns
    if column not in EXCLUDED_COLUMNS
]

STATIC_CATEGORICAL_COLUMNS = [
    column
    for column in ["station_id"]
    if column in train_df.columns
]

# ============================================================
# Prepare Data Types
# ============================================================

for df in [train_df, val_df, test_df]:

    df["station_id"] = df["station_id"].astype(str)
    df["sequence_segment_id"] = (
        df["sequence_segment_id"]
        .astype(str)
    )

    df["time_idx"] = (
        pd.to_numeric(
            df["time_idx"],
            errors="raise",
        )
        .astype(np.int64)
    )

    for column in NUMERIC_FEATURE_COLUMNS:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        ).astype(np.float32)

# ============================================================
# Validation
# ============================================================

missing_summary = pd.DataFrame({
    "train_missing": train_df[NUMERIC_FEATURE_COLUMNS].isna().sum(),
    "validation_missing": val_df[NUMERIC_FEATURE_COLUMNS].isna().sum(),
    "test_missing": test_df[NUMERIC_FEATURE_COLUMNS].isna().sum(),
})

print("Feature configuration completed.")
print(f"Numeric model features : {len(NUMERIC_FEATURE_COLUMNS)}")
print(f"Static categoricals    : {STATIC_CATEGORICAL_COLUMNS}")
print(f"Target columns         : {len(TARGET_COLUMNS)}")

print("\nRemaining missing values:")
display(
    missing_summary[
        missing_summary.sum(axis=1) > 0
    ]
)

Feature configuration completed.
Numeric model features : 33
Static categoricals    : ['station_id']
Target columns         : 9

Remaining missing values:


,train_missing,validation_missing,test_missing
ambient_temperature,0,0,10
hours_since_previous,66,0,0


# Fit train only Normalization statistics and validate sequecne

In [4]:
# ============================================================
# Train-Only Normalization Statistics
# ============================================================

normalization_columns = [
    column
    for column in NUMERIC_FEATURE_COLUMNS
    if column != "time_idx"
]

normalization_stats = {}

for column in normalization_columns:

    mean_value = float(train_df[column].mean())
    std_value = float(train_df[column].std())

    if not np.isfinite(std_value) or std_value == 0:
        std_value = 1.0

    normalization_stats[column] = {
        "mean": mean_value,
        "std": std_value,
    }

# ============================================================
# Sequence Validation
# ============================================================

REQUIRED_SEQUENCE_LENGTH = (
    ENCODER_LENGTH + PREDICTION_LENGTH
)

sequence_summary = {}

for split_name, df in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():

    segment_lengths = (
        df.groupby(
            ["station_id", "sequence_segment_id"],
            observed=True,
        )
        .size()
    )

    eligible_segments = (
        segment_lengths >= REQUIRED_SEQUENCE_LENGTH
    ).sum()

    estimated_windows = (
        segment_lengths
        .sub(REQUIRED_SEQUENCE_LENGTH)
        .add(1)
        .clip(lower=0)
        .sum()
    )

    sequence_summary[split_name] = {
        "rows": int(len(df)),
        "stations": int(df["station_id"].nunique()),
        "segments": int(len(segment_lengths)),
        "eligible_segments": int(eligible_segments),
        "estimated_windows": int(estimated_windows),
    }

    print(f"\n{split_name.upper()}")
    print(f"Segments          : {len(segment_lengths):,}")
    print(f"Eligible segments : {eligible_segments:,}")
    print(f"Estimated windows : {estimated_windows:,}")

assert sequence_summary["train"]["estimated_windows"] > 0
assert sequence_summary["validation"]["estimated_windows"] > 0
assert sequence_summary["test"]["estimated_windows"] > 0

print("\nTrain-only normalization statistics fitted.")
print("Sequence validation passed.")


TRAIN
Segments          : 4,795
Eligible segments : 1,675
Estimated windows : 1,559,445

VALIDATION
Segments          : 1,995
Eligible segments : 694
Estimated windows : 560,498

TEST
Segments          : 2,238
Eligible segments : 642
Estimated windows : 479,204

Train-only normalization statistics fitted.
Sequence validation passed.


# Savepreprocessing metadata and final validation

In [5]:
# ============================================================
# Save Preprocessing Metadata
# ============================================================

preprocessing_metadata = {
    "encoder_length": ENCODER_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "required_sequence_length": REQUIRED_SEQUENCE_LENGTH,
    "target_columns": TARGET_COLUMNS,
    "weather_columns": WEATHER_COLUMNS,
    "numeric_feature_columns": NUMERIC_FEATURE_COLUMNS,
    "static_categorical_columns": STATIC_CATEGORICAL_COLUMNS,
    "normalization_columns": normalization_columns,
    "normalization_stats": normalization_stats,
    "sequence_summary": sequence_summary,
    "split_metadata": split_metadata,
}

with open(PREPROCESSOR_PATH, "wb") as file:
    pickle.dump(
        preprocessing_metadata,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

# ============================================================
# Final Validation
# ============================================================

assert PREPROCESSOR_PATH.exists()

with open(PREPROCESSOR_PATH, "rb") as file:
    saved_metadata = pickle.load(file)

assert saved_metadata["encoder_length"] == 168
assert saved_metadata["prediction_length"] == 15
assert saved_metadata["target_columns"] == TARGET_COLUMNS

print("=" * 65)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 65)

print(f"\nTrain rows      : {len(train_df):,}")
print(f"Validation rows : {len(val_df):,}")
print(f"Test rows       : {len(test_df):,}")

print(f"\nNumeric features : {len(NUMERIC_FEATURE_COLUMNS)}")
print(f"Targets          : {len(TARGET_COLUMNS)}")
print(f"Encoder length   : {ENCODER_LENGTH}")
print(f"Forecast horizon : {PREDICTION_LENGTH}")

print("\nEstimated Forecast Windows")

for split_name, summary in sequence_summary.items():
    print(
        f"{split_name.capitalize():<12}: "
        f"{summary['estimated_windows']:,}"
    )

print(f"\nMetadata saved to:\n{PREPROCESSOR_PATH}")

print("\nReady for TFT and PatchTST dataset/model creation.")

DATA PREPROCESSING COMPLETED SUCCESSFULLY

Train rows      : 2,007,399
Validation rows : 747,056
Test rows       : 674,665

Numeric features : 33
Targets          : 9
Encoder length   : 168
Forecast horizon : 15

Estimated Forecast Windows
Train       : 1,559,445
Validation  : 560,498
Test        : 479,204

Metadata saved to:
../Data/Processed/preprocessing_metadata.pkl

Ready for TFT and PatchTST dataset/model creation.
